# Etapa 3 · Carga do Data Warehouse - star schema `dw_campo_real`

**Campo Real Energia · Data Warehouse** · continuação dos notebooks de PLD, Open-Meteo e qualidade

Este notebook **executa os scripts SQL versionados** da pasta `sql/` (o modelo e as decisões estão no documento de modelagem) e **valida** a carga:

| Script | O que faz |
|---|---|
| `01_ddl_dw_campo_real.sql` | cria o schema, dimensões, fatos e views (**apaga** o schema se existir) |
| `02_carga_dimensoes.sql` | tempo, submercado, tipo de parada, origem do dado |
| `03_scd2_dim_usina.sql` | `dim_usina` com SCD Tipo 2 + Tipo 1 (preserva histórico) |
| `04_carga_fatos.sql` | 4 fatos + resultados de qualidade; escolhe a versão da usina vigente em cada data |

O DW é lido **somente** das tabelas tratadas e do cadastro de submercados e feriados. O painel lê **só** do `dw_campo_real`.

In [1]:
%pip install -q psycopg2-binary pyyaml pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Configuração e conexão

In [3]:
import os, time, getpass
from pathlib import Path
import yaml, numpy as np, pandas as pd, psycopg2

CONFIG_PATH = Path("config.yaml")
BASE_YAML = """
banco:
  variavel_ambiente: NEON_DATABASE_URL
  arquivo_env: .env
"""
DW_YAML = """
dw:
  sql_dir: sql
  recriar_dw: false      # true roda o 01 (APAGA o schema, inclusive o histórico SCD2)
  demo_scd2: false       # true simula uma mudança de potência dentro de uma transação e desfaz
"""

def mesclar(padrao, atual):
    for k, v in (atual or {}).items():
        if isinstance(v, dict) and isinstance(padrao.get(k), dict):
            mesclar(padrao[k], v)
        else:
            padrao[k] = v
    return padrao

cfg = yaml.safe_load(BASE_YAML + DW_YAML)
if CONFIG_PATH.exists():
    atual = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    if "dw" not in atual:
        with open(CONFIG_PATH, "a", encoding="utf-8") as f:
            f.write("\n" + DW_YAML.strip() + "\n")
        print("Seção 'dw' adicionada ao config.yaml existente.")
    cfg = mesclar(cfg, atual)
else:
    CONFIG_PATH.write_text((BASE_YAML + DW_YAML).strip() + "\n", encoding="utf-8")

B, D = cfg["banco"], cfg["dw"]
SQL_DIR = Path(D["sql_dir"])

def ler_url_banco():
    var = B["variavel_ambiente"]
    if os.environ.get(var):
        return os.environ[var]
    env = Path(B["arquivo_env"])
    if env.exists():
        for linha in env.read_text(encoding="utf-8").splitlines():
            linha = linha.strip()
            if linha and not linha.startswith("#") and "=" in linha:
                k, v = linha.split("=", 1)
                if k.strip() == var:
                    return v.strip().strip('"').strip("'")
    return getpass.getpass(f"Cole a connection string do Neon ({var}): ")

conn = psycopg2.connect(ler_url_banco(), connect_timeout=30,
                        keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=5)

def consulta(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    conn.commit()
    return pd.DataFrame(rows, columns=cols)

def escalar(sql):
    return consulta(sql).iloc[0, 0]

def rodar_sql(nome):
    sql = (SQL_DIR / nome).read_text(encoding="utf-8")
    t0 = time.time()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
    except Exception:
        conn.rollback()
        print(f"ERRO em {nome}")
        raise
    with conn.cursor() as cur:
        cur.execute("SET search_path TO public")
    conn.commit()
    print(f"{nome}: ok ({time.time() - t0:.1f}s)")

# pré-requisitos: camada tratada da Etapa 2 e tabelas externas
necessarias = ["qualidade.usina_tratada", "qualidade.parada_tratada", "qualidade.medicao_geracao_tratada",
               "qualidade.metricas_qualidade", "qualidade.impacto_negocio", "externo.pld_horario",
               "externo.clima_horario", "externo.clima_ponto", "campo_real_energia.submercado",
               "campo_real_energia.fonte_energia", "campo_real_energia.feriado_nacional"]
faltam = [t for t in necessarias if escalar(f"SELECT to_regclass('{t}')") is None]
if faltam:
    raise RuntimeError(f"Faltam tabelas: {faltam}. Rode antes os notebooks anteriores (PLD, Open-Meteo, qualidade).")
for f in ["01_ddl_dw_campo_real.sql", "02_carga_dimensoes.sql", "03_scd2_dim_usina.sql", "04_carga_fatos.sql"]:
    assert (SQL_DIR / f).exists(), f"script ausente: {SQL_DIR / f}"
print("Pré-requisitos ok.")

Pré-requisitos ok.


## 2. Criação do schema (DDL)

O script 01 **apaga** o schema `dw_campo_real`. Por isso ele só roda se o schema ainda **não existir** ou se `dw.recriar_dw` for `true` no `config.yaml`. Em reexecuções normais, pule direto para as cargas.

In [6]:
existe = escalar("SELECT to_regnamespace('dw_campo_real')") is not None
if not existe or D["recriar_dw"]:
    rodar_sql("01_ddl_dw_campo_real.sql")
else:
    print("Schema dw_campo_real já existe; DDL não executado (dw.recriar_dw = false).")

01_ddl_dw_campo_real.sql: ok (0.2s)


## 3. Carga: dimensões, SCD2 e fatos

In [11]:
rodar_sql("02_carga_dimensoes.sql")
rodar_sql("03_scd2_dim_usina.sql")
rodar_sql("04_carga_fatos.sql")

02_carga_dimensoes.sql: ok (0.3s)
03_scd2_dim_usina.sql: ok (0.1s)
04_carga_fatos.sql: ok (4.3s)


## 4. Validações da carga

Cada linha compara um valor esperado (calculado nas tabelas de origem da camada tratada) com o que foi para o DW. `OK` = igual. A reconciliação de receita e de geração confronta o DW com o **mesmo número que a Etapa 2 gravou** em `qualidade.impacto_negocio`.

In [12]:
verif = []
def checar(nome, esperado, obtido, tol=0.0):
    ok = (abs(float(esperado) - float(obtido)) <= tol)
    verif.append((nome, esperado, obtido, "OK" if ok else "FALHA"))

S = "dw_campo_real"
checar("dim_tempo: 17.520 horas + membro desconhecido", 17521, escalar(f"SELECT COUNT(*) FROM {S}.dim_tempo"))
checar("dim_tempo: sk_tempo contíguo (máx = 17.520)", 17520, escalar(f"SELECT MAX(sk_tempo) FROM {S}.dim_tempo"))
checar("dim_usina: 1 versão atual por usina + desconhecido",
       escalar("SELECT COUNT(*) FROM qualidade.usina_tratada") + 1,
       escalar(f"SELECT COUNT(*) FROM {S}.dim_usina WHERE registro_atual"))
checar("fato_geracao = medição tratada",
       escalar("SELECT COUNT(*) FROM qualidade.medicao_geracao_tratada"),
       escalar(f"SELECT COUNT(*) FROM {S}.fato_geracao_horaria"))
checar("fato_pld = PLD tratado",
       escalar("SELECT COUNT(*) FROM externo.pld_horario"),
       escalar(f"SELECT COUNT(*) FROM {S}.fato_pld_horario"))
checar("fato_clima = clima tratado",
       escalar("SELECT COUNT(*) FROM externo.clima_horario"),
       escalar(f"SELECT COUNT(*) FROM {S}.fato_clima_horario"))
checar("fato_indisponibilidade = horas distintas (usina, hora) em parada",
       escalar(f"""SELECT COUNT(*) FROM (SELECT DISTINCT p.id_usina, t.sk_tempo
                   FROM qualidade.parada_tratada p
                   JOIN {S}.dim_tempo t ON t.data_hora BETWEEN date_trunc('hour', p.inicio) AND p.fim) x"""),
       escalar(f"SELECT COUNT(*) FROM {S}.fato_indisponibilidade_horaria"))

desc = escalar(f"""SELECT
   (SELECT COUNT(*) FROM {S}.fato_geracao_horaria WHERE sk_usina = -1 OR sk_submercado = -1 OR sk_origem_dado = -1)
 + (SELECT COUNT(*) FROM {S}.fato_pld_horario WHERE sk_submercado = -1)
 + (SELECT COUNT(*) FROM {S}.fato_clima_horario WHERE sk_usina = -1 OR sk_submercado = -1)
 + (SELECT COUNT(*) FROM {S}.fato_indisponibilidade_horaria WHERE sk_usina = -1 OR sk_submercado = -1 OR sk_tipo_parada = -1)""")
checar("fatos apontando para membro 'desconhecido' (esperado 0)", 0, desc)

checar("geração total DW = Etapa 2 (depois)",
       escalar("SELECT depois FROM qualidade.impacto_negocio WHERE nivel='total' AND indicador='geracao_mwh'"),
       escalar(f"SELECT SUM(energia_mwh) FROM {S}.fato_geracao_horaria"), tol=0.01)
checar("receita valorada DW = Etapa 2 (depois)",
       escalar("SELECT depois FROM qualidade.impacto_negocio WHERE nivel='total' AND indicador='receita_rs'"),
       escalar(f"SELECT SUM(receita_rs) FROM {S}.vw_geracao_valorada"), tol=1.0)

verif = pd.DataFrame(verif, columns=["verificação", "esperado", "obtido", "resultado"])
display(verif)
print("Todas as verificações OK" if (verif["resultado"] == "OK").all() else "ATENÇÃO: há verificações com FALHA")

,verificação,esperado,obtido,resultado
0,dim_tempo: 17.520 horas + membro desconhecido,1.752100e+04,17521,OK
1,dim_tempo: sk_tempo contíguo (máx = 17.520),1.752000e+04,17520,OK
2,dim_usina: 1 versão atual por usina + desconhe...,1.000000e+01,10,OK
3,fato_geracao = medição tratada,7.866300e+04,78663,OK
4,fato_pld = PLD tratado,6.499200e+04,64992,OK
5,fato_clima = clima tratado,7.884000e+04,78840,OK
6,fato_indisponibilidade = horas distintas (usin...,1.672000e+03,1672,OK
7,fatos apontando para membro 'desconhecido' (es...,0.000000e+00,0,OK
8,geração total DW = Etapa 2 (depois),3.118869e+06,3118868.656,OK
9,receita valorada DW = Etapa 2 (depois),6.376470e+08,637646971.46959,OK


Todas as verificações OK


In [13]:
print("Linhas por tabela do DW:")
display(consulta(f"""
    SELECT 'dim_tempo' AS tabela, COUNT(*) AS linhas FROM {S}.dim_tempo
    UNION ALL SELECT 'dim_usina', COUNT(*) FROM {S}.dim_usina
    UNION ALL SELECT 'dim_submercado', COUNT(*) FROM {S}.dim_submercado
    UNION ALL SELECT 'dim_tipo_parada', COUNT(*) FROM {S}.dim_tipo_parada
    UNION ALL SELECT 'dim_origem_dado', COUNT(*) FROM {S}.dim_origem_dado
    UNION ALL SELECT 'fato_geracao_horaria', COUNT(*) FROM {S}.fato_geracao_horaria
    UNION ALL SELECT 'fato_pld_horario', COUNT(*) FROM {S}.fato_pld_horario
    UNION ALL SELECT 'fato_clima_horario', COUNT(*) FROM {S}.fato_clima_horario
    UNION ALL SELECT 'fato_indisponibilidade_horaria', COUNT(*) FROM {S}.fato_indisponibilidade_horaria
    UNION ALL SELECT 'fato_qualidade_regra', COUNT(*) FROM {S}.fato_qualidade_regra
    UNION ALL SELECT 'fato_impacto_limpeza', COUNT(*) FROM {S}.fato_impacto_limpeza"""))

print("\nDimensão de tempo: conferência dos calendários (2025):")
display(consulta(f"""
    SELECT patamar_carga, COUNT(*) AS horas,
           COUNT(*) FILTER (WHERE feriado) AS horas_em_feriado,
           COUNT(*) FILTER (WHERE dia_util) AS horas_dia_util
    FROM {S}.dim_tempo WHERE ano = 2025 GROUP BY 1 ORDER BY 1"""))

Linhas por tabela do DW:


,tabela,linhas
0,dim_usina,10
1,fato_qualidade_regra,66
2,dim_submercado,5
3,fato_impacto_limpeza,64
4,dim_tipo_parada,3
5,dim_origem_dado,3
6,fato_indisponibilidade_horaria,1672
7,dim_tempo,17521
8,fato_pld_horario,64992
9,fato_clima_horario,78840



Dimensão de tempo: conferência dos calendários (2025):


,patamar_carga,horas,horas_em_feriado,horas_dia_util
0,Leve,3609,312,1764
1,Média,4395,0,3528
2,Pesada,756,0,756


## 5. O modelo responde às perguntas do painel?

Consultas **somente no DW**, provar que o modelo consegue responder antes de abrir o Power BI/Tableau.

In [14]:
print("P1/P2/P3 · geração, receita valorada ao PLD, preço capturado e razão de captura por fonte (2025):")
display(consulta(f"""
    WITH pm AS (
        SELECT p.sk_submercado, AVG(p.pld_rs_mwh) AS pld_medio
        FROM {S}.fato_pld_horario p JOIN {S}.dim_tempo t ON t.sk_tempo = p.sk_tempo
        WHERE t.ano = 2025 GROUP BY 1)
    SELECT u.cod_fonte,
           ROUND(SUM(v.energia_mwh), 1)                                       AS geracao_mwh,
           ROUND(SUM(v.receita_rs) / 1e6, 2)                                  AS receita_mi_rs,
           ROUND(SUM(v.receita_rs) / NULLIF(SUM(v.energia_mwh), 0), 2)        AS preco_capturado,
           ROUND(SUM(v.receita_rs) / NULLIF(SUM(v.energia_mwh * pm.pld_medio), 0), 3) AS razao_captura
    FROM {S}.vw_geracao_valorada v
    JOIN {S}.dim_usina u ON u.sk_usina = v.sk_usina
    JOIN pm ON pm.sk_submercado = v.sk_submercado
    GROUP BY u.cod_fonte ORDER BY receita_mi_rs DESC"""))

print("\nP4 · PLD médio por patamar de carga e faixa horária (2025):")
display(consulta(f"""
    SELECT t.patamar_carga, t.faixa_horaria, ROUND(AVG(p.pld_rs_mwh), 2) AS pld_medio, COUNT(*) AS horas
    FROM {S}.fato_pld_horario p JOIN {S}.dim_tempo t ON t.sk_tempo = p.sk_tempo
    WHERE t.ano = 2025 GROUP BY 1, 2 ORDER BY 1, 2"""))

print("\nP5 · perda por paradas, por tipo (energia e receita ESTIMADAS):")
display(consulta(f"""
    SELECT tp.tipo,
           SUM(f.horas_indisponiveis)                          AS horas_indisponiveis,
           ROUND(SUM(f.energia_nao_gerada_estimada_mwh), 1)    AS energia_estimada_mwh,
           ROUND(SUM(f.receita_nao_gerada_estimada_rs) / 1e6, 3) AS receita_estimada_mi_rs,
           COUNT(*) FILTER (WHERE NOT f.tem_referencia)        AS horas_sem_referencia
    FROM {S}.fato_indisponibilidade_horaria f
    JOIN {S}.dim_tipo_parada tp ON tp.sk_tipo_parada = f.sk_tipo_parada
    GROUP BY tp.tipo ORDER BY tp.tipo"""))

print("\nP5 · detalhe por usina (atenção às horas sem referência):")
display(consulta(f"""
    SELECT u.cod_usina, f.id_parada_origem, tp.tipo,
           SUM(f.horas_indisponiveis) AS horas,
           COUNT(*) FILTER (WHERE f.tem_referencia) AS horas_com_referencia,
           ROUND(SUM(f.energia_nao_gerada_estimada_mwh), 1) AS energia_estimada_mwh,
           ROUND(SUM(f.receita_nao_gerada_estimada_rs), 0)  AS receita_estimada_rs
    FROM {S}.fato_indisponibilidade_horaria f
    JOIN {S}.dim_usina u ON u.sk_usina = f.sk_usina
    JOIN {S}.dim_tipo_parada tp ON tp.sk_tipo_parada = f.sk_tipo_parada
    GROUP BY u.cod_usina, f.id_parada_origem, tp.tipo ORDER BY u.cod_usina, f.id_parada_origem"""))

print("\nP6 · clima x geração (correlação por usina, 2025; só para conferir que os fatos se cruzam):")
display(consulta(f"""
    SELECT u.cod_usina, u.cod_fonte,
           ROUND(corr(g.energia_mwh::float8, c.radiacao_w_m2::float8)::numeric, 3) AS corr_radiacao,
           ROUND(corr(g.energia_mwh::float8, c.vento_100m_ms::float8)::numeric, 3) AS corr_vento100
    FROM {S}.fato_geracao_horaria g
    JOIN {S}.fato_clima_horario c ON c.sk_tempo = g.sk_tempo AND c.sk_usina = g.sk_usina
    JOIN {S}.dim_usina u ON u.sk_usina = g.sk_usina
    WHERE u.cod_fonte IN ('UFV', 'EOL') AND g.energia_mwh IS NOT NULL
    GROUP BY u.cod_usina, u.cod_fonte ORDER BY u.cod_fonte, u.cod_usina"""))

print("\nP7 · índice de qualidade por dimensão (lido do DW):")
display(consulta(f"""
    SELECT dimensao_qualidade,
           MAX(CASE WHEN fase = 'ANTES'  THEN indice END) AS antes,
           MAX(CASE WHEN fase = 'DEPOIS' THEN indice END) AS depois
    FROM (SELECT fase, dimensao_qualidade, ROUND(AVG(indice_qualidade_pct), 2) AS indice
          FROM {S}.vw_indice_qualidade GROUP BY 1, 2) x
    GROUP BY dimensao_qualidade ORDER BY dimensao_qualidade"""))

P1/P2/P3 · geração, receita valorada ao PLD, preço capturado e razão de captura por fonte (2025):


,cod_fonte,geracao_mwh,receita_mi_rs,preco_capturado,razao_captura
0,UHE,1513225.1,304.72,201.37,0.954
1,EOL,629178.1,129.90,206.46,1.099
2,UTE,380879.9,87.28,229.16,1.253
3,BIO,183739.7,44.18,240.43,1.072
4,UFV,281056.0,42.69,151.89,0.775
5,PCH,130789.9,28.88,220.80,0.973



P4 · PLD médio por patamar de carga e faixa horária (2025):


,patamar_carga,faixa_horaria,pld_medio,horas
0,Leve,Madrugada,217.26,8760
1,Leve,Manhã,138.57,2700
2,Leve,Noite,232.64,1488
3,Leve,Tarde,119.13,1488
4,Média,Manhã,162.80,6060
5,Média,Noite,245.33,4248
6,Média,Tarde,203.90,7272
7,Pesada,Noite,262.29,3024



P5 · perda por paradas, por tipo (energia e receita ESTIMADAS):


,tipo,horas_indisponiveis,energia_estimada_mwh,receita_estimada_mi_rs,horas_sem_referencia
0,Forçada,88,2761.1,0.687,0
1,Programada,1584,47712.7,11.233,408


... (2 outputs left out)


P6 · clima x geração (correlação por usina, 2025; só para conferir que os fatos se cruzam):


,cod_usina,cod_fonte,corr_radiacao,corr_vento100
0,CRE-EOL-003,EOL,-0.475,0.221
1,CRE-EOL-007,EOL,-0.351,0.257
2,CRE-UFV-004,UFV,0.930,-0.412
3,CRE-UFV-006,UFV,0.936,-0.430



P7 · índice de qualidade por dimensão (lido do DW):


,dimensao_qualidade,antes,depois
0,Acurácia,99.88,99.92
1,Atualidade,97.12,97.13
2,Completude,97.49,99.98
3,Consistência,96.97,100.00
4,Integridade referencial,99.99,100.00
5,Unicidade,96.65,100.00
6,Validade,96.18,100.00


## 6. Demonstração do SCD Tipo 2

Com `dw.demo_scd2: true` no `config.yaml`, a célula abaixo **simula uma repotenciação** da `CRE-UFV-006` dentro de uma transação, reroda o script 03 e mostra as duas versões da usina. No fim faz **ROLLBACK**: nada muda no banco.

In [ ]:
if D["demo_scd2"]:
    sql03 = (SQL_DIR / "03_scd2_dim_usina.sql").read_text(encoding="utf-8")
    try:
        with conn.cursor() as cur:
            cur.execute("UPDATE qualidade.usina_tratada SET potencia_mw = potencia_mw + 10 WHERE cod_usina = 'CRE-UFV-006'")
            cur.execute(sql03)
            cur.execute("""SELECT sk_usina, cod_usina, potencia_mw, dt_inicio_vigencia, dt_fim_vigencia, registro_atual
                           FROM dw_campo_real.dim_usina WHERE cod_usina = 'CRE-UFV-006' ORDER BY dt_inicio_vigencia""")
            cols = [d[0] for d in cur.description]
            print("Versões de CRE-UFV-006 depois da mudança simulada de potência (+10 MW):")
            display(pd.DataFrame(cur.fetchall(), columns=cols))
    finally:
        conn.rollback()
        with conn.cursor() as cur:
            cur.execute("SET search_path TO public")
        conn.commit()
    print("ROLLBACK feito: o banco continua como antes da demonstração.")
else:
    print("Demonstração desligada (dw.demo_scd2: false).")